Copied from https://docs.carbonplan.org/ocr/en/latest/how-to/work-with-data.html.

# Work with data

This guide shows how to access Open Climate Risk output data using [Icechunk](https://icechunk.io/en/stable/), a versioned data format for cloud-native geospatial data.


## Raster / Xarray


### Import required libraries


In [1]:
import icechunk
import xarray as xr

### Connect to the Icechunk repository

Production fire risk data is stored in an Icechunk repository on S3. We'll connect to version `v1.1.0` of the wind-adjusted fire risk output. For valid versions, check out the [GitHub releases page](https://github.com/carbonplan/ocr/releases).


In [ ]:
# Configure S3 storage for the Icechunk repository
version = "v1.1.0"
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix=f"carbonplan/carbonplan-ocr/output/fire-risk/tensor/production/{version}/ocr.icechunk",
    region="us-west-2",
    anonymous=True,
)

# Open the repository
repo = icechunk.Repository.open(storage)

# Create a read-only session on the main branch
session = repo.readonly_session("main")

### Open the dataset with xarray


In [ ]:
# Open the dataset
ds = xr.open_dataset(session.store, engine="zarr", chunks={})
ds

<xarray.Dataset> Size: 652GB
Dimensions:        (latitude: 97579, longitude: 208881)
Coordinates:
  * latitude       (latitude) float64 781kB 22.43 22.43 22.43 ... 52.48 52.48
  * longitude      (longitude) float64 2MB -128.4 -128.4 ... -64.05 -64.05
Data variables:
    bp_2011        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2011_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 82GB dask.array<chunksize=(6000, 4500), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/

### Explore the data variables

The dataset contains wind-adjusted fire risk metrics. Let's examine the available variables:


In [ ]:
# List all data variables
print("Data variables:")
for var in ds.data_vars:
    print(f"  - {var}: {ds[var].attrs.get('long_name', 'No description')}")

Data variables:
  - bp_2011: No description
  - bp_2011_riley: No description
  - bp_2047_riley: No description
  - rps_2047: No description
  - rps_scott: No description
  - crps_scott: No description
  - bp_2047: No description
  - rps_2011: No description


### Select a spatial subset

Extract data for a specific geographic region using coordinate slicing:


In [5]:
# Example: Select data for California region
california_subset = ds.sel(
    latitude=slice(42, 32),  # Southern to Northern California
    longitude=slice(-125, -114),  # Western to Eastern California
)

california_subset

<xarray.Dataset> Size: 286kB
Dimensions:        (latitude: 0, longitude: 35715)
Coordinates:
  * latitude       (latitude) float64 0B 
  * longitude      (longitude) float64 286kB -125.0 -125.0 ... -114.0 -114.0
Data variables:
    bp_2011        (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    bp_2011_riley  (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 0B dask.array<chunksize=(0, 2501), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/